# AI Exposure Drift Monitor — Walkthrough

This notebook demonstrates AEDM's end-to-end analysis pipeline: from raw organizational data to exposure scores, drift detection, demographic disparity analysis, and reskilling urgency rankings.

**What is AEDM?** AEDM operationalizes the exposure framework from Anthropic's March 2026 paper, *"Labor Market Impacts of AI: A New Measure and Early Evidence"* (Massenkoff & McCrory), into a workforce planning tool. It computes how much of each role's work is exposed to AI — both theoretically (what AI *could* do) and observationally (what AI *is* doing) — and tracks how that exposure changes over time.

**What you'll learn:**
1. How to load and validate organizational role data
2. How exposure scores are computed and what they mean
3. How to detect drift in exposure over multiple quarters
4. How to identify demographic groups with disproportionate exposure
5. How to prioritize reskilling investment using composite urgency scores
6. How to generate reports and visualizations

## 1. Setup

First, let's import the modules we need and suppress structlog's info-level output so our notebook stays clean.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import logging
import structlog
structlog.configure(
    wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING),
)

from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)

# AEDM modules
from aedm.ingest import parse_csv, load_reference_rates, load_quarterly_snapshots, map_roles
from aedm.analysis import (
    compute_org_exposure, org_mean_exposure, exposure_by_department,
    analyze_all_disparities, analyze_gender_disparity,
    analyze_education_disparity, analyze_pay_band_disparity,
    score_org_urgency, detect_org_drift,
)
from aedm.output import (
    scores_to_dataframe,
    exposure_heatmap, exposure_distribution,
    demographic_disparity_bars, urgency_matrix, drift_sparklines,
    generate_markdown_report,
)

print("AEDM loaded successfully.")

AEDM loaded successfully.


## 2. Load and Validate Organizational Data

AEDM accepts a CSV with at minimum a `title` column. Richer analysis is possible when you include SOC codes, departments, headcounts, and demographic fields.

We'll use the bundled sample data for a fictional company, **Acme Corp** (200 roles across 10 departments).

In [2]:
DATA_DIR = Path("../data")

# Parse the CSV — this validates the schema automatically
roles = parse_csv(DATA_DIR / "sample" / "acme_corp_roles.csv")
print(f"Loaded {len(roles)} roles")
print(f"Departments: {sorted(set(r.department for r in roles))}")
print(f"Total headcount: {sum(r.headcount for r in roles):,}")

Loaded 200 roles
Departments: ['Engineering', 'Executive', 'Finance', 'HR', 'IT', 'Legal', 'Marketing', 'Operations', 'R&D', 'Sales']
Total headcount: 842


### SOC Code Mapping

Each role needs a Standard Occupational Classification (SOC) code to look up its reference exposure rates. If your data doesn't include SOC codes, AEDM maps job titles automatically using fuzzy matching.

In [3]:
# Map roles to SOC codes (no-op for roles that already have them)
roles = map_roles(roles)

# Show a few examples
mapping_examples = [(r.title, r.soc_code, r.soc_major_group) for r in roles[:8]]
pd.DataFrame(mapping_examples, columns=["Title", "SOC Code", "Major Group"])

,Title,SOC Code,Major Group
0,Software Engineer I,15-1252,15-0000
1,Software Engineer II,15-1252,15-0000
2,Senior Software Engineer,15-1252,15-0000
3,Staff Software Engineer,15-1252,15-0000
4,Principal Software Engineer,15-1252,15-0000
5,Software Engineering Manager,11-9041,11-0000
6,Director of Engineering,11-9041,11-0000
7,VP of Engineering,11-9041,11-0000


### Load Reference Exposure Rates

Reference rates come from Anthropic's published research. Each SOC major group has a **theoretical** exposure rate (what AI could do) and an **observed** exposure rate (what AI is actually doing). The gap between them is the "uncovered area" — tasks ripe for future automation.

In [4]:
rates = load_reference_rates(DATA_DIR / "reference" / "anthropic_exposure_rates.json")
print(f"Loaded reference rates for {len(rates)} SOC major groups\n")

# Show rates sorted by theoretical exposure
rates_df = pd.DataFrame([
    {
        "SOC Group": code,
        "Name": rate.group_name,
        "Theoretical": rate.theoretical_exposure,
        "Observed": rate.observed_exposure,
        "Gap": rate.coverage_gap,
    }
    for code, rate in sorted(rates.items())
])
rates_df.sort_values("Theoretical", ascending=False).head(10)

Loaded reference rates for 22 SOC major groups



,SOC Group,Name,Theoretical,Observed,Gap
1,13-0000,Business and Financial Operations,0.9430,0.2000,0.7430
2,15-0000,Computer and Mathematical,0.9400,0.3300,0.6100
6,23-0000,Legal,0.9350,0.2200,0.7150
0,11-0000,Management,0.9130,0.1800,0.7330
4,19-0000,"Life, Physical, and Social Science",0.8950,0.1700,0.7250
16,43-0000,Office and Administrative Support,0.8850,0.2100,0.6750
3,17-0000,Architecture and Engineering,0.8740,0.1500,0.7240
7,25-0000,Educational Instruction and Library,0.8700,0.1400,0.7300
8,27-0000,"Arts, Design, Entertainment, Sports, and Media",0.8520,0.1900,0.6620
5,21-0000,Community and Social Service,0.8200,0.0900,0.7300


## 3. Compute Exposure Scores

The core metric is the **blended exposure index**: a weighted combination of theoretical and observed exposure.

```
exposure_index = 0.4 × theoretical + 0.6 × observed
```

Observed exposure gets more weight because workforce planning should be grounded in what AI *is* doing, not just what it *could* do. But theoretical exposure signals where adoption is likely to expand next.

In [5]:
# Compute exposure for all roles
scores = compute_org_exposure(roles, rates)

# Organization-level summary
mean_exp = org_mean_exposure(roles, scores)
print(f"Organization mean exposure: {mean_exp:.1%}")
print(f"Roles analyzed: {len(scores)}")

# Tier distribution
from collections import Counter
tier_counts = Counter(s.tier.value for s in scores)
for tier in ["Critical", "High", "Moderate", "Low"]:
    print(f"  {tier}: {tier_counts.get(tier, 0)} roles")

Organization mean exposure: 50.4%
Roles analyzed: 200
  Critical: 0 roles
  High: 56 roles
  Moderate: 136 roles
  Low: 8 roles


### Top 10 Most Exposed Roles

In [6]:
# Build a combined DataFrame and show the top 10
df = scores_to_dataframe(roles, scores)
df = df.sort_values("blended_exposure", ascending=False)

top_10 = df[["title", "department", "headcount", "theoretical_exposure",
             "observed_exposure", "blended_exposure", "exposure_tier"]].head(10)
top_10.index = range(1, 11)
top_10.index.name = "Rank"
top_10

,title,department,headcount,theoretical_exposure,observed_exposure,blended_exposure,exposure_tier
Rank,,,,,,,
1,Software Engineer I,Engineering,35,0.9400,0.3300,0.5740,High
2,QA Automation Engineer,Engineering,6,0.9400,0.3300,0.5740,High
3,Senior Security Engineer,IT,2,0.9400,0.3300,0.5740,High
4,IT Support Specialist,IT,8,0.9400,0.3300,0.5740,High
5,IT Systems Administrator,IT,5,0.9400,0.3300,0.5740,High
6,Network Administrator,IT,3,0.9400,0.3300,0.5740,High
7,Database Administrator,IT,4,0.9400,0.3300,0.5740,High
8,SOC Analyst,IT,3,0.9400,0.3300,0.5740,High
9,Penetration Tester,IT,2,0.9400,0.3300,0.5740,High


### Exposure by Department

Department-level aggregation uses headcount-weighted means, so a department with many high-exposure roles and large headcount will score higher than one with a single high-exposure specialist.

In [7]:
dept_means = exposure_by_department(roles, scores)

dept_df = pd.DataFrame([
    {"Department": dept, "Mean Exposure": mean}
    for dept, mean in sorted(dept_means.items(), key=lambda x: -x[1])
])
dept_df.index = range(1, len(dept_df) + 1)
dept_df

,Department,Mean Exposure
1,IT,0.5690
2,Engineering,0.5550
3,R&D,0.5442
4,Legal,0.5027
5,HR,0.4948
6,Finance,0.4903
7,Executive,0.4767
8,Marketing,0.4745
9,Operations,0.4276
10,Sales,0.4188


## 4. Visualize Exposure

AEDM includes Plotly-based interactive charts. In a Jupyter environment, these render inline with hover tooltips.

In [8]:
fig = exposure_distribution(scores, title="Acme Corp — Exposure Score Distribution")
fig.update_layout(height=400, width=800)
fig.show()

In [9]:
fig = exposure_heatmap(roles, scores, title="Acme Corp — Exposure by Department")
fig.update_layout(height=600, width=900)
fig.show()

### Reading the Heatmap

The horizontal bar chart above shows individual roles grouped by department, colored by tier:
- **Red** = Critical (75-100%): these roles have the majority of their tasks exposed to AI
- **Orange** = High (50-74%): significant exposure warranting active planning
- **Teal** = Moderate (25-49%): some task exposure, worth periodic review
- **Gray** = Low (0-24%): limited AI exposure under current technology

## 5. Drift Detection

Single-point analysis tells you *where you are*. Drift detection tells you *where you're heading*.

AEDM uses **CUSUM (Cumulative Sum) changepoint detection** with **permutation-based significance testing** to identify when exposure trends shift. It also fits a linear trend to estimate the rate of change.

We'll load four quarterly snapshots and track department-level exposure over time.

In [10]:
from collections import defaultdict

# Load quarterly snapshots
snapshots = load_quarterly_snapshots(DATA_DIR / "sample" / "acme_corp_quarterly")
print(f"Loaded {len(snapshots)} quarterly snapshots:")
for snap in snapshots:
    print(f"  {snap.period_label}: {len(snap.roles)} roles")

Loaded 4 quarterly snapshots:
  q1_2025: 99 roles
  q2_2025: 99 roles
  q3_2025: 101 roles
  q4_2025: 104 roles


In [11]:
import numpy as np

# Compute department-level exposure for each quarter
dept_time_series = defaultdict(list)

for snap in snapshots:
    snap_roles = map_roles(snap.roles)
    snap_scores = compute_org_exposure(snap_roles, rates)
    dept_means = exposure_by_department(snap_roles, snap_scores)
    for dept, mean in dept_means.items():
        dept_time_series[dept].append(mean)

# Show the time series
quarters = [s.period_label for s in snapshots]
ts_df = pd.DataFrame(dept_time_series, index=quarters).T
ts_df.columns = [q.replace("_", " ").upper() for q in quarters]
ts_df["Change"] = ts_df.iloc[:, -1] - ts_df.iloc[:, 0]
ts_df = ts_df.sort_values("Change", ascending=False)
ts_df

,Q1 2025,Q2 2025,Q3 2025,Q4 2025,Change
Marketing,0.4770,0.4770,0.4848,0.4872,0.0102
Engineering,0.5545,0.5556,0.5565,0.5573,0.0028
Finance,0.4918,0.4920,0.4924,0.4932,0.0014
Sales,0.4122,0.4122,0.4124,0.4134,0.0011
R&D,0.5574,0.5584,0.5589,0.5577,0.0003
Executive,0.4800,0.4800,0.4800,0.4800,0.0000
IT,0.5740,0.5740,0.5740,0.5740,-0.0000
HR,0.4955,0.4957,0.4955,0.4954,-0.0002
Legal,0.5038,0.5038,0.5038,0.5026,-0.0012
Operations,0.4323,0.4308,0.4298,0.4262,-0.0060


In [12]:
# Run CUSUM drift detection on department time series
rng = np.random.default_rng(42)
dept_drift = detect_org_drift(dict(dept_time_series), rng=rng)

print(f"Analyzed {len(dept_drift)} departments for drift\n")
for d in sorted(dept_drift, key=lambda x: abs(x.trend_slope), reverse=True):
    sig = "***" if d.p_value < 0.05 else ""
    print(f"  {d.entity_id:15s} | {d.direction.value:14s} | "
          f"slope={d.trend_slope:+.6f}/quarter | p={d.p_value:.3f} {sig}")

Analyzed 10 departments for drift

  Marketing       | Stable         | slope=+0.003832/quarter | p=0.585 
  Operations      | Stable         | slope=-0.001907/quarter | p=0.926 
  Engineering     | Stable         | slope=+0.000932/quarter | p=0.665 
  Finance         | Stable         | slope=+0.000441/quarter | p=0.662 
  Legal           | Stable         | slope=-0.000355/quarter | p=1.000 
  Sales           | Stable         | slope=+0.000353/quarter | p=1.000 
  R&D             | Stable         | slope=+0.000161/quarter | p=0.658 
  HR              | Stable         | slope=-0.000068/quarter | p=0.349 
  IT              | Stable         | slope=-0.000000/quarter | p=0.365 
  Executive       | Stable         | slope=+0.000000/quarter | p=1.000 


### Interpreting Drift Results

With only 4 quarterly data points, statistical power is limited — this is expected. The CUSUM test is conservative by design: it avoids false positives at the cost of missing small shifts. As more quarters accumulate, genuine trends will emerge with stronger significance.

Key departments to watch:
- **Engineering** and **Marketing** show upward exposure trends (more tasks being automated quarter-over-quarter)
- **Operations** shows a downward trend (exposure may be stabilizing as initial AI adoption matures)

The `detect_drift_cusum` function supports customizable thresholds, permutation counts, and significance levels for organizations with different risk tolerances.

In [13]:
fig = drift_sparklines(dept_drift, title="Acme Corp — Department Exposure Drift")
fig.update_layout(height=450, width=800)
fig.show()

## 6. Demographic Disparity Analysis

Anthropic's research found that AI exposure systematically skews toward female workers, more educated workers, and higher-paid workers. AEDM checks whether these macro patterns hold within your specific organization.

A segment is **flagged** when its disparity ratio exceeds **1.2x** the org-wide mean — meaning workers in that group face at least 20% more AI exposure than average.

In [14]:
# Run all disparity analyses
all_segments = analyze_all_disparities(roles, scores)

# Display as a table
seg_df = pd.DataFrame([
    {
        "Type": s.segment_type.replace("_", " ").title(),
        "Segment": s.segment_value,
        "Mean Exposure": s.mean_exposure,
        "Disparity Ratio": s.disparity_ratio,
        "Headcount": s.headcount,
        "Flagged": "Yes" if s.flagged else "",
    }
    for s in all_segments
])
seg_df

,Type,Segment,Mean Exposure,Disparity Ratio,Headcount,Flagged
0,Gender,Female,0.4954,0.9834,318,
1,Gender,Male,0.5089,1.0101,524,
2,Education,Associate,0.4706,0.9341,59,
3,Education,Bachelor,0.5149,1.0220,619,
4,Education,Doctorate,0.4965,0.9856,18,
5,Education,High School,0.3601,0.7148,49,
6,Education,Master,0.5273,1.0466,97,
7,Pay Band,$0K-$50K,0.3973,0.7887,106,
8,Pay Band,$120K-$160K,0.5465,1.0848,134,
9,Pay Band,$160K-$200K,0.5359,1.0637,38,


In [15]:
fig = demographic_disparity_bars(all_segments, title="Acme Corp — Exposure Disparity by Demographic Segment")
fig.update_layout(height=500, width=900)
fig.show()

### Reading the Disparity Chart

The dashed red line marks the 1.2x flag threshold. Segments above this line warrant closer investigation:
- **Are certain pay bands or education levels concentrated in high-exposure departments?**
- **Does the organization's gender distribution create uneven AI exposure risk?**

Note: disparity ratios describe *correlation*, not *causation*. A high ratio means more exposure in that segment, not that exposure occurs *because of* demographic characteristics.

## 7. Reskilling Urgency Scoring

Exposure alone doesn't determine where to invest in reskilling. A role with 80% exposure and 2 employees is a different priority than one with 60% exposure and 200 employees.

AEDM's urgency score is a composite of four factors:

| Component | Weight | What It Captures |
|-----------|--------|-----------------|
| Exposure level | 30% | How much of the role is AI-exposed |
| Drift velocity | 25% | How fast exposure is changing |
| Headcount at risk | 25% | How many people are affected (log-scaled) |
| Reskill difficulty | 20% | How few lower-exposure occupations are available to transition into |

Since we're doing single-period analysis here, drift is set to 0 and the remaining weights are rescaled proportionally.

In [16]:
# Score reskilling urgency for all roles
urgency_scores = score_org_urgency(roles, scores, None, rates)

# Top 15 most urgent
role_map = {r.role_id: r for r in roles}
urgency_rows = []
for i, u in enumerate(urgency_scores[:15], 1):
    role = role_map[u.role_id]
    urgency_rows.append({
        "Rank": i,
        "Title": role.title,
        "Department": role.department,
        "Headcount": role.headcount,
        "Urgency Score": u.score,
        "Tier": u.tier.value,
    })

urgency_df = pd.DataFrame(urgency_rows).set_index("Rank")
urgency_df

,Title,Department,Headcount,Urgency Score,Tier
Rank,,,,,
1,Sales Representative,Sales,20,0.4950,Moderate
2,Software Engineer I,Engineering,35,0.4884,Moderate
3,Sales Development Representative,Sales,15,0.4753,Moderate
4,Financial Analyst,Finance,10,0.4737,Moderate
5,Software Engineer II,Engineering,28,0.4728,Moderate
6,Customer Support Specialist,Operations,12,0.4661,Moderate
7,Product Manager,Engineering,8,0.4623,Moderate
8,Account Executive,Sales,12,0.4603,Moderate
9,Accountant,Finance,8,0.4592,Moderate


In [17]:
fig = urgency_matrix(roles, scores, urgency_scores, title="Acme Corp — Reskilling Urgency Matrix")
fig.update_layout(height=550, width=850)
fig.show()

### Reading the Urgency Matrix

Each bubble represents a role. The X-axis is exposure, the Y-axis is urgency, and the bubble size is headcount. Roles in the **upper-right** quadrant are the highest priority for reskilling investment: they're highly exposed, urgently scored, and affect many people.

The scatter also reveals roles where exposure is high but urgency is lower — often because headcount is small or reskilling options are plentiful.

## 8. Generate a Full Report

AEDM can produce a complete Markdown report suitable for sharing with leadership. This combines all the analyses above into a structured document.

In [18]:
dept_means = exposure_by_department(roles, scores)

report = generate_markdown_report(
    roles=roles,
    scores=scores,
    org_mean=mean_exp,
    dept_means=dept_means,
    urgency_scores=urgency_scores,
    demographic_segments=all_segments,
)

# Show first 60 lines of the report
print("\n".join(report.split("\n")[:60]))
print("\n... [report continues] ...")

# AI Exposure Analysis Report

*Generated by AEDM — AI Exposure Drift Monitor*

## Executive Summary

- **Total roles analyzed:** 200
- **Total headcount:** 842
- **Organization mean exposure:** 50.4%

### Exposure Distribution by Tier

| Tier | Roles | Headcount | % of Workforce |
|------|-------|-----------|----------------|
| 🔴 Critical | 0 | 0 | 0.0% |
| 🟠 High | 56 | 367 | 43.6% |
| 🟡 Moderate | 136 | 448 | 53.2% |
| 🟢 Low | 8 | 27 | 3.2% |

## Department Exposure

| Department | Mean Exposure | Tier |
|------------|---------------|------|
| IT | 56.9% | 🟠 High |
| Engineering | 55.5% | 🟠 High |
| R&D | 54.4% | 🟠 High |
| Legal | 50.3% | 🟠 High |
| HR | 49.5% | 🟡 Moderate |
| Finance | 49.0% | 🟡 Moderate |
| Executive | 47.7% | 🟡 Moderate |
| Marketing | 47.5% | 🟡 Moderate |
| Operations | 42.8% | 🟡 Moderate |
| Sales | 41.9% | 🟡 Moderate |

## Top 20 Most Exposed Roles

| Rank | Title | Department | Exposure | Tier | Headcount |
|------|-------|------------|----------|------|----

### Exporting Results

For integration with other tools, AEDM exports all computed metrics as structured CSV or JSON:

```python
from aedm.output import export_csv, export_json

export_csv(roles, scores, Path("output/exposure_scores.csv"), urgency_scores)
export_json(roles, scores, Path("output/exposure_scores.json"),
            urgency_scores=urgency_scores,
            demographic_segments=all_segments,
            org_mean=mean_exp, dept_means=dept_means)
```

## Next Steps

This walkthrough covered the core AEDM pipeline. Here's where to go from here:

**For workforce planners:**
- Replace the sample CSV with your organization's actual role data
- Run quarterly analysis to build a drift baseline
- Use the Streamlit dashboard for interactive exploration: `aedm dashboard --input your_roles.csv`

**For data scientists:**
- Review the [methodology documentation](../docs/methodology.md) for statistical details
- Customize exposure weights, significance levels, and disparity thresholds via `AEDMSettings`
- Extend the reference rates with organization-specific observed exposure data

**For more information:**
- [Quick Start Guide](../docs/quickstart.md) — 5-minute setup
- [Data Dictionary](../docs/data_dictionary.md) — field definitions
- [Architecture](../ARCHITECTURE.md) — system design
- Anthropic's research: *"Labor Market Impacts of AI: A New Measure and Early Evidence"* (Massenkoff & McCrory, March 2026)